<a href="https://colab.research.google.com/github/caiohamamura/forestsat2026_rgedi_workshop/blob/main/rgedi_incolab_version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛰️ Introduction to NASA's GEDI and rGEDI
## An R Package for Accessing, Handling, and Processing GEDI Data

---

**Authors:** Dr. Inácio Bueno &nbsp;·&nbsp; Dr. Caio Hamamura

---

### 🌍 About this Workshop

Welcome! This notebook guides you through NASA's **Global Ecosystem Dynamics Investigation (GEDI)** mission and the `rGEDI` package in R. By the end, you'll be able to:

| Step | What You'll Do |
|------|----------------|
| 📦 **Setup** | Install and load all required packages |
| 🗺️ **Define** | Set your study area and date range |
| 🔍 **Search & Download** | Find and download GEDI L1B, L2A, and L2B files |
| ✂️ **Clip & Plot** | Clip data by bounding box or polygon geometry |
| 📡 **Waveforms** | Extract and visualize GEDI full waveforms |
| 🌲 **Metrics** | Explore canopy height and vegetation biophysics |
| 📊 **Statistics** | Compute polygon-based and gridded summaries |

---

> **💡 Tip:** Run each cell in order from top to bottom. If you see an error, check whether the previous cells have been run successfully.

## 📦 Section 1 — Package Installation & Setup

---

### What happens here?

This section checks for all required R packages and installs any that are missing.
`rGEDI` is installed directly from GitHub because it is not on CRAN.

| Package | Purpose |
|---------|------------------------------------------|
| `rGEDI` | Core package: read, clip, and process GEDI data |
| `sf` | Read and manipulate vector geometries (shapefiles) |
| `terra` / `raster` | Raster operations and gridded data |
| `leaflet` | Interactive map visualizations |
| `ggplot2` | Static plots |
| `rasterVis` | Enhanced raster visualization |
| `viridis` | Perceptually uniform colour palettes |

> ⏳ **Note:** The first run may take a few minutes while packages are compiled from source.

In [ ]:
rm(list = ls())
gc()

# List of required CRAN packages
packages <- c("remotes", "sf", "leaflet", "ggplot2", "terra", "raster", "rasterVis", "viridis")

# Identify and install any missing CRAN packages
missing_pkg <- packages[!sapply(packages, requireNamespace, quietly = TRUE)]
if (length(missing_pkg) > 0) {
  install.packages(missing_pkg)
}

# Install rGEDI from GitHub if not already installed
if (!requireNamespace("rGEDI", quietly = TRUE)) {
  remotes::install_github(
    "carlos-alberto-silva/rGEDI",
    dependencies = TRUE
  )
}

# Load all required packages
library(remotes)
library(rGEDI)
library(sf)
library(leaflet)
library(ggplot2)
library(terra)
library(raster)
library(rasterVis)
library(viridis)

cat("✅ All packages loaded successfully!\n")

## 🗺️ Section 2 — Define Study Area & Parameters

---

### What happens here?

You define **where** and **when** to search for GEDI data.

- **Bounding box** — A rectangle defined by upper-left and lower-right corners in geographic coordinates (WGS84 / EPSG:4326)
- **Shapefile** — A polygon file used for precise geometry-based clipping (must contain a field named `id`)
- **Date range** — The temporal window for your GEDI search
- **Output directory** — Where the downloaded `.h5` files will be saved

```
  ul_lon ──────────┐
  ul_lat           │
                   │   ← Bounding Box
                   │
                   └──────────── lr_lon
                                  lr_lat
```

> **✏️ Customize:** Change the bounding box coordinates, shapefile path, and dates to match your own study area.

In [ ]:
# ----- Output directory (edit this path to your Google Drive or working folder) -----
outdir <- "/content/drive/MyDrive/GEDI_Workshop"  # Change this path as needed
dir.create(outdir, showWarnings = FALSE, recursive = TRUE)
cat("📁 Output directory:\n", outdir, "\n")

# ----- Bounding box coordinates (WGS84 / EPSG:4326) -----
# ul = upper left  |  lr = lower right
ul_lat <- 29.77
lr_lat <- 29.73
ul_lon <- -82.24
lr_lon <- -82.19

# ----- Shapefile for geometry-based clipping -----
polygon_filepath <- "aoi.shp"  # Must contain a field named 'id'

# ----- Date range for GEDI data search -----
daterange <- c("2020-01-01", "2020-05-31")

cat("\n🌐 Bounding box:\n")
cat("  Upper-left  (lat, lon):", ul_lat, ",", ul_lon, "\n")
cat("  Lower-right (lat, lon):", lr_lat, ",", lr_lon, "\n")
cat("\n📅 Date range:", daterange[1], "→", daterange[2], "\n")

## 🔍 Section 3 — Search & Download GEDI Data

---

### What happens here?

The `gedifinder()` function queries NASA's GEDI data archive and returns a list of `.h5` file URLs that intersect your study area and date range.

We search for three GEDI data products:

| Product | Code | Contents |
|---------|------|----------|
| **Level 1B** | `GEDI01_B` | Geolocated waveforms |
| **Level 2A** | `GEDI02_A` | Elevation & canopy height metrics |
| **Level 2B** | `GEDI02_B` | Canopy cover & vertical profile metrics |

---

### ⚠️ Workshop Note

To keep download time manageable, **only the 2nd file** from each product is downloaded. In a full analysis, you would download and process all returned files.

> **🔐 Authentication:** `gediDownload()` requires a NASA Earthdata account. If prompted, enter your credentials. [Register here for free →](https://urs.earthdata.nasa.gov/)

In [ ]:
# ----- Search for GEDI files -----
cat("🔍 Searching for GEDI files...\n")

gLevel1B <- gedifinder(
  product = "GEDI01_B", ul_lat = ul_lat, ul_lon = ul_lon,
  lr_lat = lr_lat, lr_lon = lr_lon, version = "002", daterange = daterange
)

gLevel2A <- gedifinder(
  product = "GEDI02_A", ul_lat = ul_lat, ul_lon = ul_lon,
  lr_lat = lr_lat, lr_lon = lr_lon, version = "002", daterange = daterange
)

gLevel2B <- gedifinder(
  product = "GEDI02_B", ul_lat = ul_lat, ul_lon = ul_lon,
  lr_lat = lr_lat, lr_lon = lr_lon, version = "002", daterange = daterange
)

cat("\n📋 Files found:\n")
cat("  GEDI Level 1B:", length(gLevel1B), "files\n")
cat("  GEDI Level 2A:", length(gLevel2A), "files\n")
cat("  GEDI Level 2B:", length(gLevel2B), "files\n")

In [ ]:
# ----- Keep only the 2nd file of each product (for workshop speed) -----
if (length(gLevel1B) >= 2) gLevel1B <- gLevel1B[2] else gLevel1B <- character(0)
if (length(gLevel2A) >= 2) gLevel2A <- gLevel2A[2] else gLevel2A <- character(0)
if (length(gLevel2B) >= 2) gLevel2B <- gLevel2B[2] else gLevel2B <- character(0)

# ----- Download GEDI files -----
cat("⬇️  Downloading GEDI files (this may take several minutes)...\n\n")

if (length(gLevel1B) > 0) {
  cat("  Downloading Level 1B...\n")
  gediDownload(filepath = gLevel1B, outdir = outdir)
} else {
  cat("  ⚠️  No Level 1B files found for this area and date range.\n")
}

if (length(gLevel2A) > 0) {
  cat("  Downloading Level 2A...\n")
  gediDownload(filepath = gLevel2A, outdir = outdir)
} else {
  cat("  ⚠️  No Level 2A files found for this area and date range.\n")
}

if (length(gLevel2B) > 0) {
  cat("  Downloading Level 2B...\n")
  gediDownload(filepath = gLevel2B, outdir = outdir)
} else {
  cat("  ⚠️  No Level 2B files found for this area and date range.\n")
}

cat("\n✅ Download complete!\n")

## ✂️ Section 4 — Read, Clip & Plot GEDI Data

---

### What happens here?

GEDI `.h5` files contain data for entire orbital tracks — far more than your study area. This section demonstrates **two spatial clipping approaches**:

| Clipping Method | Function Pattern | Use Case |
|-----------------|-----------------|----------|
| **Bounding box** | `clipLevel*(..., xmin, xmax, ymin, ymax)` | Quick rectangular filter |
| **Polygon geometry** | `clipLevel*Geometry(..., polygon)` | Precise boundary matching |

The diagram below illustrates the difference:

```
  ┌─────────────────────────────┐  ← Full tile (entire orbit)
  │  · · · · · · · · · · · · · │
  │  · · ┌──────────┐ · · · · ·│  ← Bounding box clip
  │  · · │  ╭────╮  │ · · · · ·│
  │  · · │  │ AOI│  │ · · · · ·│  ← Polygon geometry clip
  │  · · │  ╰────╯  │ · · · · ·│
  │  · · └──────────┘ · · · · ·│
  │  · · · · · · · · · · · · · │
  └─────────────────────────────┘
```

> The **geometry-clipped** result is used as the main dataset in all subsequent steps.

In [ ]:
# ----- List all downloaded HDF5 files -----
gedi_files <- list.files(path = outdir, pattern = "\\.h5$", full.names = TRUE)

cat("📂 Downloaded GEDI files found:\n")
print(basename(gedi_files))

# ----- Separate files by product -----
level1b_files <- gedi_files[grepl("GEDI01_B", basename(gedi_files))]
level2a_files <- gedi_files[grepl("GEDI02_A", basename(gedi_files))]
level2b_files <- gedi_files[grepl("GEDI02_B", basename(gedi_files))]

# ----- Read all files into lists -----
cat("\n📖 Reading GEDI files into memory...\n")
gedilevel1b_list <- lapply(level1b_files, readLevel1B)
names(gedilevel1b_list) <- basename(level1b_files)

gedilevel2a_list <- lapply(level2a_files, readLevel2A)
names(gedilevel2a_list) <- basename(level2a_files)

gedilevel2b_list <- lapply(level2b_files, readLevel2B)
names(gedilevel2b_list) <- basename(level2b_files)

cat("✅ Files loaded:\n")
cat("  Level 1B:", length(gedilevel1b_list), "file(s)\n")
cat("  Level 2A:", length(gedilevel2a_list), "file(s)\n")
cat("  Level 2B:", length(gedilevel2b_list), "file(s)\n")

In [ ]:
# ----- Bounding box coordinates (derived from Section 2) -----
xmin <- min(ul_lon, lr_lon)
xmax <- max(ul_lon, lr_lon)
ymin <- min(ul_lat, lr_lat)
ymax <- max(ul_lat, lr_lat)

# ----- Read and validate the polygon shapefile -----
polygon_sf <- read_sf(file.path(outdir, polygon_filepath))
split_by   <- "id"  # Field used as the polygon identifier

if (!split_by %in% names(polygon_sf)) {
  stop(paste0("The polygon layer does not contain the field '", split_by, "'."))
}

if (is.na(st_crs(polygon_sf))) {
  stop("The polygon shapefile does not have a defined coordinate reference system.")
}

# Reproject to geographic coordinates if needed
if (st_crs(polygon_sf)$epsg != 4326) {
  polygon_sf <- st_transform(polygon_sf, 4326)
  cat("ℹ️  Polygon reprojected to EPSG:4326.\n")
}

cat("✅ Polygon loaded with", nrow(polygon_sf), "feature(s).\n")

In [ ]:
# =====================================================================
# 4.1 GEDI Level 1B — Geolocation clipping
# =====================================================================

# -- Raw geolocation (full tile) --
level1bGeo_raw_list <- lapply(seq_along(gedilevel1b_list), function(i) {
  geo <- getLevel1BGeo(level1b = gedilevel1b_list[[i]], select = c("elevation_bin0"))
  geo$source_file <- names(gedilevel1b_list)[i]
  geo$shot_number <- as.character(geo$shot_number)
  geo
})
level1bGeo_raw <- do.call(rbind, level1bGeo_raw_list)

# -- Bounding box clip --
level1bGeo_bbox_list <- lapply(level1bGeo_raw_list, function(geo) {
  geo_clip <- clipLevel1BGeo(geo, xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax)
  if (!is.null(geo_clip) && nrow(geo_clip) > 0) geo_clip else NULL
})
level1bGeo_bbox_list <- Filter(Negate(is.null), level1bGeo_bbox_list)
if (length(level1bGeo_bbox_list) == 0) stop("No Level 1B shots inside the bounding box.")
level1bGeo_bbox <- do.call(rbind, level1bGeo_bbox_list)
level1bGeo_bbox$shot_number <- as.character(level1bGeo_bbox$shot_number)

# -- Polygon geometry clip --
level1bGeo_geom_list <- lapply(level1bGeo_raw_list, function(geo) {
  geo_clip <- clipLevel1BGeoGeometry(geo, polygon = polygon_sf, split_by = split_by)
  if (!is.null(geo_clip) && nrow(geo_clip) > 0) geo_clip else NULL
})
level1bGeo_geom_list <- Filter(Negate(is.null), level1bGeo_geom_list)
if (length(level1bGeo_geom_list) == 0) {
  warning("No Level 1B shots inside the shapefile geometry.")
  level1bGeo_geom <- NULL
} else {
  level1bGeo_geom <- do.call(rbind, level1bGeo_geom_list)
  level1bGeo_geom$shot_number <- as.character(level1bGeo_geom$shot_number)
}

# Use geometry-clipped data as the primary Level 1B object
level1bGeo <- level1bGeo_geom

# ----- Quick checks -----
cat("📡 Level 1B coordinate ranges:\n")
cat("  Full tile   — lon:", round(range(level1bGeo_raw$longitude_bin0,  na.rm=TRUE), 4),
    "| lat:", round(range(level1bGeo_raw$latitude_bin0,  na.rm=TRUE), 4), "\n")
cat("  BBox clip   — lon:", round(range(level1bGeo_bbox$longitude_bin0, na.rm=TRUE), 4),
    "| lat:", round(range(level1bGeo_bbox$latitude_bin0, na.rm=TRUE), 4), "\n")
cat("  Geom clip   — lon:", round(range(level1bGeo_geom$longitude_bin0, na.rm=TRUE), 4),
    "| lat:", round(range(level1bGeo_geom$latitude_bin0, na.rm=TRUE), 4), "\n")

In [ ]:
# =====================================================================
# 4.2 GEDI Level 2A — Elevation & height metrics clipping
# =====================================================================

# -- Raw metrics (full tile) --
level2AM_raw_list <- lapply(seq_along(gedilevel2a_list), function(i) {
  l2a <- getLevel2AM(gedilevel2a_list[[i]])
  l2a$source_file <- names(gedilevel2a_list)[i]
  l2a$shot_number <- as.character(l2a$shot_number)
  l2a
})
level2AM_raw <- do.call(rbind, level2AM_raw_list)

# -- Bounding box clip --
level2AM_bbox_list <- lapply(level2AM_raw_list, function(l2a) {
  l2a_clip <- clipLevel2AM(l2a, xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax)
  if (!is.null(l2a_clip) && nrow(l2a_clip) > 0) l2a_clip else NULL
})
level2AM_bbox_list <- Filter(Negate(is.null), level2AM_bbox_list)
if (length(level2AM_bbox_list) == 0) {
  warning("No Level 2A shots inside the bounding box.")
  level2AM_bbox <- NULL
} else {
  level2AM_bbox <- do.call(rbind, level2AM_bbox_list)
  level2AM_bbox$shot_number <- as.character(level2AM_bbox$shot_number)
}

# -- Polygon geometry clip --
level2AM_geom_list <- lapply(level2AM_raw_list, function(l2a) {
  l2a_clip <- clipLevel2AMGeometry(l2a, polygon = polygon_sf, split_by = split_by)
  if (!is.null(l2a_clip) && nrow(l2a_clip) > 0) l2a_clip else NULL
})
level2AM_geom_list <- Filter(Negate(is.null), level2AM_geom_list)
if (length(level2AM_geom_list) == 0) {
  warning("No Level 2A shots inside the shapefile geometry.")
  level2AM_geom <- NULL
} else {
  level2AM_geom <- do.call(rbind, level2AM_geom_list)
  level2AM_geom$shot_number <- as.character(level2AM_geom$shot_number)
}

# Use geometry-clipped data as the primary Level 2A object
level2AM <- level2AM_geom
cat("✅ Level 2A clipped:", nrow(level2AM), "shots retained.\n")

In [ ]:
# =====================================================================
# 4.3 GEDI Level 2B — Canopy cover & profile metrics clipping
# =====================================================================

# -- Raw metrics (full tile) --
level2BVPM_raw_list <- lapply(seq_along(gedilevel2b_list), function(i) {
  l2b <- getLevel2BVPM(gedilevel2b_list[[i]])
  l2b$source_file <- names(gedilevel2b_list)[i]
  l2b$shot_number <- as.character(l2b$shot_number)
  l2b
})
level2BVPM_raw <- do.call(rbind, level2BVPM_raw_list)

# -- Bounding box clip --
level2BVPM_bbox_list <- lapply(level2BVPM_raw_list, function(l2b) {
  l2b_clip <- clipLevel2BVPM(l2b, xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax)
  if (!is.null(l2b_clip) && nrow(l2b_clip) > 0) l2b_clip else NULL
})
level2BVPM_bbox_list <- Filter(Negate(is.null), level2BVPM_bbox_list)
if (length(level2BVPM_bbox_list) == 0) {
  warning("No Level 2B shots inside the bounding box.")
  level2BVPM_bbox <- NULL
} else {
  level2BVPM_bbox <- do.call(rbind, level2BVPM_bbox_list)
  level2BVPM_bbox$shot_number <- as.character(level2BVPM_bbox$shot_number)
}

# -- Polygon geometry clip --
level2BVPM_geom_list <- lapply(level2BVPM_raw_list, function(l2b) {
  l2b_clip <- clipLevel2BVPMGeometry(l2b, polygon = polygon_sf, split_by = split_by)
  if (!is.null(l2b_clip) && nrow(l2b_clip) > 0) l2b_clip else NULL
})
level2BVPM_geom_list <- Filter(Negate(is.null), level2BVPM_geom_list)
if (length(level2BVPM_geom_list) == 0) {
  warning("No Level 2B shots inside the shapefile geometry.")
  level2BVPM_geom <- NULL
} else {
  level2BVPM_geom <- do.call(rbind, level2BVPM_geom_list)
  level2BVPM_geom$shot_number <- as.character(level2BVPM_geom$shot_number)
}

# Use geometry-clipped data as the primary Level 2B object
level2BVPM <- level2BVPM_geom
cat("✅ Level 2B clipped:", nrow(level2BVPM), "shots retained.\n")

In [ ]:
# =====================================================================
# 4.4 Interactive map — compare tile, bounding-box, and AOI footprints
# =====================================================================

# Build bounding box polygon for display
bbox_sf <- st_as_sfc(
  st_bbox(c(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax), crs = st_crs(4326))
)

# 10 km buffer around bbox (for plotting nearby tile footprints)
bbox_buffer_10km <- bbox_sf %>% st_buffer(10000)

# Convert raw Level 1B to sf and filter to plotting area
level1bGeo_raw_sf <- st_as_sf(
  level1bGeo_raw, coords = c("longitude_bin0", "latitude_bin0"), crs = 4326, remove = FALSE
)
level1bGeo_raw_plot <- st_intersection(level1bGeo_raw_sf, bbox_buffer_10km)

# Build interactive leaflet map
leaflet() %>%
  addProviderTiles(providers$Esri.WorldImagery) %>%
  addPolygons(data = bbox_sf, color = "cyan",   weight = 2, fill = FALSE, group = "Bounding box") %>%
  addPolygons(data = polygon_sf, color = "yellow", weight = 2, fill = FALSE, group = "Shapefile") %>%
  addCircleMarkers(
    data = level1bGeo_raw_plot, lng = ~longitude_bin0, lat = ~latitude_bin0,
    radius = 2, stroke = FALSE, fillOpacity = 0.9, color = "red",   group = "Tile footprints"
  ) %>%
  addCircleMarkers(
    data = level1bGeo_bbox, lng = ~longitude_bin0, lat = ~latitude_bin0,
    radius = 2, stroke = FALSE, fillOpacity = 0.9, color = "cyan",  group = "BBox footprints"
  ) %>%
  addCircleMarkers(
    data = level1bGeo_geom, lng = ~longitude_bin0, lat = ~latitude_bin0,
    radius = 2, stroke = FALSE, fillOpacity = 0.9, color = "yellow", group = "AOI footprints"
  ) %>%
  addLayersControl(
    overlayGroups = c("Bounding box", "Shapefile", "Tile footprints", "BBox footprints", "AOI footprints"),
    options = layersControlOptions(collapsed = FALSE)
  ) %>%
  addLegend(
    position = "bottomright",
    colors = c("red", "cyan", "yellow"),
    labels = c("Tile footprints", "BBox footprints", "AOI footprints"),
    title = "GEDI footprints"
  ) %>%
  fitBounds(lng1 = xmin, lat1 = ymin, lng2 = xmax, lat2 = ymax)

## 📡 Section 5 — GEDI Level 1B: Geolocated Waveforms

---

### What happens here?

GEDI Level 1B data contains the **raw return energy waveforms** — the direct echo of the laser pulse reflected from the surface and vegetation.

A waveform records how much laser energy was returned at each height interval:

```
Elevation (m)
    ▲
 30 │        ╭──╮             ← Top of canopy
 20 │       ╭╯  ╰╮
 10 │      ╭╯    ╰──╮         ← Mid canopy / understory
  0 │─────╭╯        ╰─────╮  ← Ground return
    └──────────────────────────► Waveform amplitude
```

**5.1** converts the clipped data to a shapefile  
**5.2** maps the footprint locations  
**5.3** plots the waveform for a specific shot

> **✏️ Customize:** Change `shot_number` below to inspect a different GEDI footprint. Valid shot numbers come from `head(level1bGeo)`.

In [ ]:
# =====================================================================
# 5.1 Export Level 1B geolocation as shapefile
# =====================================================================

# Preview the table
cat("👀 Preview of clipped Level 1B geolocation:\n")
print(head(level1bGeo))

# Convert to sf
level1bGeo_sf <- st_as_sf(
  level1bGeo, coords = c("longitude_bin0", "latitude_bin0"), crs = 4326, remove = FALSE
)

# Export shapefile (remove any pre-existing version first)
shp_out  <- file.path(outdir, "GEDI_Level1B_Geolocation.shp")
old_shps <- list.files(outdir, pattern = "^GEDI_Level1B_Geolocation\\.(shp|shx|dbf|prj|cpg)$", full.names = TRUE)
if (length(old_shps) > 0) file.remove(old_shps)

write_sf(level1bGeo_sf, shp_out, delete_layer = TRUE)
cat("\n✅ Shapefile saved:\n", shp_out, "\n")

In [ ]:
# =====================================================================
# 5.2 Interactive map of Level 1B footprints
# =====================================================================

level1b_map <- leaflet() %>%
  addProviderTiles(providers$Esri.WorldImagery) %>%
  addCircleMarkers(
    lng     = level1bGeo$longitude_bin0,
    lat     = level1bGeo$latitude_bin0,
    radius  = 1,
    opacity = 1,
    color   = "red"
  ) %>%
  addScaleBar(options = list(imperial = FALSE)) %>%
  addLegend(colors = "red", labels = "Footprints", title = "GEDI Level 1B") %>%
  setView(
    lng  = mean(level1bGeo$longitude_bin0, na.rm = TRUE),
    lat  = mean(level1bGeo$latitude_bin0,  na.rm = TRUE),
    zoom = 15
  )

print(level1b_map)

In [ ]:
# =====================================================================
# 5.3 Extract and plot a full waveform
# =====================================================================

# Check available shot numbers
cat("Available shot numbers (first 6):\n")
print(head(level1bGeo$shot_number))

# ----- ✏️ Set the shot number you want to inspect -----
shot_number <- "72530800300429393"

# Validate shot exists
if (!shot_number %in% level1bGeo$shot_number) {
  stop("❌ The selected shot_number was not found in the clipped Level 1B data.")
}

# Identify which file contains this shot and retrieve the Level 1B object
source_file_wf <- unique(level1bGeo$source_file[level1bGeo$shot_number == shot_number])[1]
wf_index       <- match(source_file_wf, names(gedilevel1b_list))

# Extract the waveform
wf <- getLevel1BWF(level1b = gedilevel1b_list[[wf_index]], shot_number = shot_number)

# Plot: absolute amplitude (left) and relative amplitude (right)
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1), cex.axis = 1.2)

plot(wf, relative = FALSE, polygon = TRUE,  type = "l", lwd = 2, col = "forestgreen",
     xlab = "Waveform amplitude",   ylab = "Elevation (m)", main = "Absolute amplitude")
grid()

plot(wf, relative = TRUE,  polygon = FALSE, type = "l", lwd = 2, col = "forestgreen",
     xlab = "Waveform amplitude (%)", ylab = "Elevation (m)", main = "Relative amplitude")
grid()

## 🌲 Section 6 — GEDI Level 2A: Elevation & Height Metrics

---

### What happens here?

GEDI Level 2A provides **elevation and canopy height metrics** derived from the waveform. Key variables include:

| Variable | Description |
|----------|-------------|
| `elev_highestreturn` | Elevation of the top of the canopy (m, WGS84 ellipsoid) |
| `elev_lowestmode` | Elevation of the ground (m, WGS84 ellipsoid) |
| `rh100` | Relative height at 100% energy — canopy top height above ground (m) |
| `rh50`, `rh75` ... | Relative heights at other energy percentiles |

---

### 📐 What are Relative Height (RH) metrics?

RH metrics measure the **height above ground** at which a given percentage of the total waveform energy has been returned:

```
  ↑ Height
  │                 ┤ rh100 = top of canopy
  │             ┤ rh75
  │         ┤ rh50  = median canopy height
  │     ┤ rh25
  └──── ground (rh0 = 0 m)
```

**6.1** exports the metrics as a shapefile  
**6.2** overlays RH metrics on the waveform plot

In [ ]:
# =====================================================================
# 6.1 Preview and export Level 2A metrics
# =====================================================================

cat("👀 Preview of clipped Level 2A metrics:\n")
print(head(level2AM[, c("beam", "shot_number", "elev_highestreturn", "elev_lowestmode", "rh100")]))

# Convert to sf
level2AM_sf <- st_as_sf(
  level2AM, coords = c("lon_lowestmode", "lat_lowestmode"), crs = 4326, remove = FALSE
)

# Export shapefile
l2a_out  <- file.path(outdir, "GEDI_Level2A_Metrics.shp")
old_l2a  <- list.files(outdir, pattern = "^GEDI_Level2A_Metrics\\.(shp|shx|dbf|prj|cpg)$", full.names = TRUE)
if (length(old_l2a) > 0) file.remove(old_l2a)

write_sf(level2AM_sf, l2a_out, delete_layer = TRUE)
cat("\n✅ Shapefile saved:\n", l2a_out, "\n")

In [ ]:
# =====================================================================
# 6.2 Plot waveform with RH metrics overlay
# =====================================================================

# Find the Level 1B and Level 2A source files for the selected shot
source_file_l1b <- unique(level1bGeo$source_file[level1bGeo$shot_number == shot_number])[1]
source_file_l2a <- unique(level2AM$source_file[level2AM$shot_number  == shot_number])[1]

l1b_index <- match(source_file_l1b, names(gedilevel1b_list))
l2a_index <- match(source_file_l2a, names(gedilevel2a_list))

par(mfrow = c(1, 1), mar = c(4, 4, 2, 1), cex.axis = 1.2)
plotWFMetrics(
  level1b     = gedilevel1b_list[[l1b_index]],
  level2a     = gedilevel2a_list[[l2a_index]],
  shot_number = shot_number,
  rh          = c(25, 50, 75, 90)   # ✏️ Change these RH percentiles as desired
)

## 🌿 Section 7 — GEDI Level 2B: Canopy Cover & Vertical Profiles

---

### What happens here?

GEDI Level 2B provides **vegetation biophysical variables** that characterise canopy structure. Key metrics include:

| Variable | Full Name | Description |
|----------|-----------|-------------|
| `cover` | Canopy Cover | Fraction of ground covered by vegetation (0–1) |
| `pai` | Plant Area Index | Total one-sided area of plant material per unit ground area |
| `fhd_normal` | Foliage Height Diversity | Shannon diversity of PAI across height layers |
| `pgap_theta` | Gap Probability | Probability of a laser pulse passing through the canopy |
| `omega` | Clumping Index | Measures how clumped or dispersed foliage is |

---

### 📊 PAI and PAVD Profiles

The **PAI profile** and **PAVD (Plant Area Volume Density) profile** describe how vegetation is distributed vertically through the canopy:

```
  Height (m)
  ▲
  30 │▉             ← PAI / PAVD at 25–30 m height layer
  25 │▉▉▉
  20 │▉▉▉▉▉         ← Dense mid-canopy layer
  15 │▉▉▉
  10 │▉▉
   5 │▉
   0 └────────────► PAI / PAVD value
```

> **✏️ Customize:** Change the `beam` variable to inspect a different GEDI beam. Run `head(level2BPAVDProfile$beam)` to see available beams.

In [ ]:
# =====================================================================
# 7.1 Preview and export Level 2B biophysical variables
# =====================================================================

cat("👀 Preview of selected Level 2B variables:\n")
print(head(level2BVPM[, c("beam", "shot_number", "pai", "fhd_normal", "omega", "pgap_theta", "cover")]))

# Convert to sf
level2BVPM_sf <- st_as_sf(
  level2BVPM, coords = c("longitude_lastbin", "latitude_lastbin"), crs = 4326, remove = FALSE
)

# Export shapefile
l2b_out  <- file.path(outdir, "GEDI_Level2B_BiophysicalVariables.shp")
old_l2b  <- list.files(outdir, pattern = "^GEDI_Level2B_BiophysicalVariables\\.(shp|shx|dbf|prj|cpg)$", full.names = TRUE)
if (length(old_l2b) > 0) file.remove(old_l2b)

write_sf(level2BVPM_sf, l2b_out, delete_layer = TRUE)
cat("\n✅ Shapefile saved:\n", l2b_out, "\n")

In [ ]:
# =====================================================================
# 7.2 Extract and plot PAI and PAVD vertical profiles
# =====================================================================

# Extract PAI profiles from all Level 2B files
level2BPAIProfile_list <- lapply(seq_along(gedilevel2b_list), function(i) {
  pai <- getLevel2BPAIProfile(gedilevel2b_list[[i]])
  pai$source_file <- names(gedilevel2b_list)[i]
  pai
})
level2BPAIProfile <- do.call(rbind, level2BPAIProfile_list)
level2BPAIProfile$shot_number <- as.character(level2BPAIProfile$shot_number)

# Extract PAVD profiles from all Level 2B files
level2BPAVDProfile_list <- lapply(seq_along(gedilevel2b_list), function(i) {
  pavd <- getLevel2BPAVDProfile(gedilevel2b_list[[i]])
  pavd$source_file <- names(gedilevel2b_list)[i]
  pavd
})
level2BPAVDProfile <- do.call(rbind, level2BPAVDProfile_list)
level2BPAVDProfile$shot_number <- as.character(level2BPAVDProfile$shot_number)

# Filter to only shots present in the clipped Level 2B metrics
if (!is.null(level2BVPM)) {
  valid_shots         <- unique(level2BVPM$shot_number)
  level2BPAIProfile   <- level2BPAIProfile[level2BPAIProfile$shot_number   %in% valid_shots, ]
  level2BPAVDProfile  <- level2BPAVDProfile[level2BPAVDProfile$shot_number %in% valid_shots, ]
}

cat("Available beams in PAVD profile data:\n")
print(head(unique(level2BPAVDProfile$beam)))

# ----- ✏️ Set the beam to inspect -----
beam <- "BEAM0101"

# Plot PAI profile
if (nrow(level2BPAIProfile) > 0 && beam %in% level2BPAIProfile$beam) {
  gPAIprofile <- plotPAIProfile(level2BPAIProfile, beam = beam, elev = TRUE)
} else {
  warning("Selected beam not found in PAI profile data.")
}

# Plot PAVD profile
if (nrow(level2BPAVDProfile) > 0 && beam %in% level2BPAVDProfile$beam) {
  gPAVDprofile <- plotPAVDProfile(level2BPAVDProfile, beam = beam, elev = TRUE)
} else {
  warning("Selected beam not found in PAVD profile data.")
}

## 📊 Section 8 — Polygon-Based Summary Statistics

---

### What happens here?

Instead of examining individual shots, we now **aggregate GEDI metrics across polygons** — for example, computing mean canopy height or maximum PAI within each forest stand or management unit.

The `polyStats*` family of functions applies any R expression to the GEDI shots that fall within each polygon:

```
  Polygon A          Polygon B
  ┌─────────┐       ┌────────────┐
  │  · · ·  │       │  · ·  · · │
  │  · · ·  │  →    │  · ·  · · │
  └─────────┘       └────────────┘
  mean(rh100) = 18m  mean(rh100) = 24m
```

You can supply **any R expression** as `func`, including custom multi-metric functions. The custom `mySetOfMetrics()` function below computes min, max, mean, and SD in a single call.

> **💡 Tip:** Set `id = "poly_id"` to get results broken down by polygon, or `id = NULL` for a single summary across all shots.

In [ ]:
# =====================================================================
# 8. Polygon-based statistics
# =====================================================================

# Custom function returning multiple statistics
mySetOfMetrics <- function(x) {
  list(
    min  = min(x,  na.rm = TRUE),
    max  = max(x,  na.rm = TRUE),
    mean = mean(x, na.rm = TRUE),
    sd   = sd(x,   na.rm = TRUE)
  )
}

# ----- 8.1 Level 2A: RH100 statistics -----
cat("🌲 Level 2A — Maximum RH100 across all shots:\n")
rh100max_st <- polyStatsLevel2AM(level2AM, func = max(rh100, na.rm = TRUE), id = NULL)
print(rh100max_st)

cat("\n🌲 Level 2A — Multiple RH100 statistics:\n")
rh100metrics_st <- polyStatsLevel2AM(level2AM, func = mySetOfMetrics(rh100), id = NULL)
print(head(rh100metrics_st))

# ----- 8.2 Level 2B: PAI and cover statistics -----
cat("\n🌿 Level 2B — Maximum PAI across all shots:\n")
pai_max <- polyStatsLevel2BVPM(level2BVPM, func = max(pai, na.rm = TRUE), id = NULL)
print(pai_max)

cat("\n🌿 Level 2B — Canopy cover statistics by polygon:\n")
cover_metrics_st <- polyStatsLevel2BVPM(level2BVPM, func = mySetOfMetrics(cover), id = "poly_id")
print(head(cover_metrics_st))

## 🗾 Section 9 — Gridded Descriptive Statistics

---

### What happens here?

Rather than aggregating to polygons, here we aggregate GEDI metrics to a **regular spatial grid** (raster), producing maps of summary statistics across the study area.

The `gridStats*` functions:
1. Overlay GEDI shot locations onto a regular grid
2. Apply a summary function to all shots in each grid cell
3. Return a multi-band `SpatRaster` (one band per statistic)

```
  GEDI shot locations        Grid statistics raster
  · · ·  · · ·  ·            ┌───┬───┬───┐
    · ·    ·   · ·    →      │18 │22 │19 │  ← mean(rh100)
  · ·  ·   · · ·             ├───┼───┼───┤
    ·   · ·  ·   ·           │20 │25 │17 │
                              └───┴───┴───┘
```

### 📐 Grid Resolution

| Resolution | Approx. ground distance (at equator) |
|------------|--------------------------------------|
| 0.001°     | ~111 m |
| 0.005°     | ~555 m |
| 0.01°      | ~1.1 km |

> **✏️ Customize:** Adjust `res` to change the grid cell size. Finer resolution (`res = 0.001`) requires more shots per cell to be meaningful.

In [ ]:
# =====================================================================
# 9.1 Gridded statistics — Level 2A RH100
# =====================================================================

cat("📊 Computing gridded statistics for RH100...\n")

# Compute gridded stats (res in decimal degrees; 0.005° ≈ 500 m)
rh100metrics <- gridStatsLevel2AM(
  level2AM = level2AM,
  func     = mySetOfMetrics(rh100),
  res      = 0.005   # ✏️ Adjust resolution here
)

# Quick preview with terra's plot()
plot(rh100metrics)

# Enhanced map with rasterVis
rh100metrics_raster <- raster::brick(rh100metrics)

rh100maps <- rasterVis::levelplot(
  rh100metrics_raster,
  layout   = c(1, 4),
  margin   = FALSE,
  xlab     = "Longitude (degrees)",
  ylab     = "Latitude (degrees)",
  colorkey = list(
    space      = "right",
    labels     = list(at = seq(0, 35, 5), font = 4),
    axis.line  = list(col = "black"),
    width      = 1
  ),
  par.settings = list(
    strip.border       = list(col = "gray"),
    strip.background   = list(col = "gray"),
    axis.line          = list(col = "gray")
  ),
  scales      = list(draw = TRUE),
  col.regions = viridis,
  at          = seq(0, 35, len = 101),
  names.attr  = c("RH100 min", "RH100 max", "RH100 mean", "RH100 SD")
)

# Display maps
print(rh100maps)

# Export as PNG
png_out <- file.path(outdir, "GEDI_Level2A_RH100_GridStats.png")
png(filename = png_out, width = 6, height = 8, units = "in", res = 300)
print(rh100maps)
dev.off()
cat("\n✅ PNG saved:\n", png_out, "\n")

In [ ]:
# =====================================================================
# 9.2 Gridded statistics — Level 2B PAI
# =====================================================================

cat("📊 Computing gridded statistics for PAI...\n")

# Replace invalid PAI fill values with NA
level2BVPM$pai[level2BVPM$pai == -9999] <- NA

# Compute gridded stats
pai_metrics <- gridStatsLevel2BVPM(
  level2BVPM = level2BVPM,
  func       = mySetOfMetrics(pai),
  res        = 0.005   # ✏️ Adjust resolution here
)

# Enhanced map with rasterVis
pai_metrics_raster <- raster::brick(pai_metrics)

pai_maps <- rasterVis::levelplot(
  pai_metrics_raster,
  layout   = c(1, 4),
  margin   = FALSE,
  xlab     = "Longitude (degrees)",
  ylab     = "Latitude (degrees)",
  colorkey = list(
    space      = "right",
    labels     = list(at = seq(0, 6, 1), font = 4),
    axis.line  = list(col = "black"),
    width      = 1
  ),
  par.settings = list(
    strip.border       = list(col = "gray"),
    strip.background   = list(col = "gray"),
    axis.line          = list(col = "gray")
  ),
  scales      = list(draw = TRUE),
  col.regions = viridis,
  at          = seq(0, 6, len = 101),
  names.attr  = c("PAI min", "PAI max", "PAI mean", "PAI SD")
)

# Display maps
print(pai_maps)

# Export as PNG
pai_png_out <- file.path(outdir, "GEDI_Level2B_PAI_GridStats.png")
png(filename = pai_png_out, width = 6, height = 8, units = "in", res = 300)
print(pai_maps)
dev.off()
cat("\n✅ PNG saved:\n", pai_png_out, "\n")

## 🎉 Workshop Complete!

---

Congratulations — you've completed the full GEDI + rGEDI workflow! Here's a summary of what you accomplished:

| ✅ | Step | Output |
|----|------|--------|
| ✅ | Installed and loaded all packages | Ready R environment |
| ✅ | Defined study area and parameters | Bounding box + shapefile |
| ✅ | Searched and downloaded GEDI data | Level 1B, 2A, 2B `.h5` files |
| ✅ | Clipped data by bbox and geometry | Filtered shot tables |
| ✅ | Visualized footprints interactively | Leaflet maps |
| ✅ | Extracted and plotted full waveforms | Waveform charts |
| ✅ | Exported metrics as shapefiles | `.shp` files in output dir |
| ✅ | Plotted PAI / PAVD vertical profiles | Canopy structure charts |
| ✅ | Computed polygon-based statistics | Summary tables |
| ✅ | Generated gridded raster maps | `.png` maps exported |

---

### 📚 Next Steps & Resources

- **rGEDI documentation:** https://github.com/carlos-alberto-silva/rGEDI
- **NASA GEDI mission:** https://gedi.umd.edu/
- **NASA Earthdata:** https://urs.earthdata.nasa.gov/
- **GEDI data products guide:** https://lpdaac.usgs.gov/product_search/?collections=GEDI

---

> 💬 **Questions?** Reach out to the workshop instructors or open an issue on the rGEDI GitHub repository.

---
*Dr. Inácio Bueno · Dr. Caio Hamamura*